### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경(부정, 중립 -> 부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test의 비율은 8:2
- Dataset을 기존의 Dataset 구성과 같이 작업
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론의 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [2]:
import re
from glob import glob
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
file_list = glob('../data/가전/*.json')
pd.read_json(file_list[0])

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
96,114667,이전부터 구매하고 싶어서 계속 눈여겨보다 구매한 락클래식이에요. 포장을 제거하고 제...,SNS,가전,영상/음향가전,엠지텍 락클래식Q9900 (정품),1,290,61,20221120,-1.0,"[{'Aspect': '품질', 'SentimentText': '마감은 좀 문제가 ..."
97,114668,요즘 집에서 작업하면서 핸드폰으로 음악을 들으니 전화를 하거나 핸드폰을 이용할때 자...,SNS,가전,영상/음향가전,오아 아이브릭 휴대용 블루투스 미니 스피커,1,338,78,20221110,-1.0,"[{'Aspect': '디자인', 'SentimentText': '디자인이 좀 그렇..."
98,114669,"처음 들어보는 생소한 브랜드의 tv라 걱정하면서 구입했는데, 역시나 후회 중입니다....",SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,333,79,20221116,-1.0,"[{'Aspect': '소음', 'SentimentText': '별로 소리를 키우지..."
99,114670,최신 기종이라고 해서 기대했는데 구기종보다 못하네요. 소재가 별로여서 예쁜 디자인이...,SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,324,74,20221116,-1.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '최신 기종..."


In [4]:
total_df = pd.DataFrame()

for file_path in file_list:
    df = pd.read_json(file_path)
    total_df = pd.concat([total_df, df], axis=0)
total_df.reset_index(drop=True, inplace=True)
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [5]:
total_df['GeneralPolarity'].value_counts()

GeneralPolarity
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [6]:
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
total_df['RawText'] = total_df['RawText'].map(normalize)

In [8]:
total_df.drop_duplicates(subset='RawText', inplace=True)
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [9]:
na_df = total_df.loc[total_df['GeneralPolarity'].isna(), ]
na_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 378 entries, 13 to 3944
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            378 non-null    int64  
 1   RawText          378 non-null    object 
 2   Source           378 non-null    object 
 3   Domain           378 non-null    object 
 4   MainCategory     378 non-null    object 
 5   ProductName      378 non-null    object 
 6   ReviewScore      378 non-null    int64  
 7   Syllable         378 non-null    int64  
 8   Word             378 non-null    int64  
 9   RDate            378 non-null    int64  
 10  GeneralPolarity  0 non-null      float64
 11  Aspects          378 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 38.4+ KB


In [10]:
total_df.dropna(inplace=True)

In [11]:
total_df['GeneralPolarity'] = total_df['GeneralPolarity'].map(lambda x: 1 if x == 1.0 else 0)

In [12]:
df = total_df[['RawText', 'GeneralPolarity']]
df

,RawText,GeneralPolarity
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요 일단 겉 부분에 ...,0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0
3,너무 별로예요 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요 음질이 거의 입 안...,0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,0
...,...,...
4051,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1
4052,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1
4053,이 제품을 추천하는 이유는요 일단 LED를 통해 공기질을 눈으로 확인할 수 있고 터...,1
4054,인터넷에서 구매하였는데요 이 공기청정기 처음 발견하고 처음에는 오잉 이게뭐지 했어요...,1


In [13]:
df.rename(columns = {'GeneralPolarity' : 'label'}, inplace=True)
df

C:\Users\student\AppData\Local\Temp\ipykernel_4860\4226988981.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns = {'GeneralPolarity' : 'label'}, inplace=True)


,RawText,label
0,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,0
1,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요 일단 겉 부분에 ...,0
2,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,0
3,너무 별로예요 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요 음질이 거의 입 안...,0
4,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,0
...,...,...
4051,사진으로는 잘 체감하지 못했는데 실제 실물을 보니 디자인이 조금 충격적이라고나 할까...,1
4052,이 에어컨의 제일 큰 장점은 조작법에 있는 것 같아요.인공지능 조작이 가능해서 한번...,1
4053,이 제품을 추천하는 이유는요 일단 LED를 통해 공기질을 눈으로 확인할 수 있고 터...,1
4054,인터넷에서 구매하였는데요 이 공기청정기 처음 발견하고 처음에는 오잉 이게뭐지 했어요...,1


In [14]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [15]:
train_df['label'].value_counts()

label
1    1776
0    1166
Name: count, dtype: int64

In [1]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

NameError: name 'SentenceTransformer' is not defined

In [ ]:
class SBERTDataset(Dataset):

    def __init__(self, document, labels):
        with torch.inference_mode():
            self.emb = sbert.encode(document, convert_to_tensor=True, normalize_embeddings=True)
        # labels를 tensor화
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        # labels의 길이를 되돌려준다
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [ ]:
train_ds = SBERTDataset(train_df['RawText'].tolist(), train_df['label'].tolist())
test_ds = SBERTDataset(train_df['RawText'].tolist(), test_df['label'].tolist())

In [ ]:
train_dl = DataLoader(train_ds, batch_size = 128, shuffle = True)
test_dl = DataLoader(test_ds, batch_size = 128, shuffle = True)

In [ ]:
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2):
        super().__init__()
        # 다중 퍼셉트론층 구성
        self.net = nn.Sequential(
            # 선형 모델
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    # 순전파 함수 -> 독립변수를 받아서 예측값을 되돌려준다
    def forward(self, x):

        result = self.net(x)
        return result

In [ ]:
in_dim = sbert.get_sentence_embedding_dimension()   # 출력 피쳐의 수를 되돌려주는 내장함수
in_dim

768

In [ ]:
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실젯값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.AdamW(clf.parameters(), lr=2e-4)

In [ ]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        # x : document데이터가 임베딩 벡터가 된 묶음(tensor)
        # y : labels 데이터가 tensor 형태 묶음
        opt.zero_grad()
        # 순전파
        logits = clf(x)
        # 손실 계산 (예측값, 실젯값)
        loss = crit(logits, y)
        # 역전파
        loss.backward()
        # 스텝
        opt.step()
        total += loss.item() * x.size(0)
    print(f'epoch : {epoch}, loss : {total/len(train_ds)}')

epoch : 0, loss : 0.6629657192265882
epoch : 1, loss : 0.6066930750695156
epoch : 2, loss : 0.5434395130761702
epoch : 3, loss : 0.4803346037297408
epoch : 4, loss : 0.42876135845074115


In [ ]:
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)    # 예측 데이터 -> [[ 0.xxxx, 0.xxxx ], [], [], ...]
        pred = logits.argmax(dim=1).tolist()    # 예측 데이터 -> [0, 1, 1, 0, ...]
        # y_true에 y를 리스트의 형태로 변환하고 데이터를 확장시킨다
        y_true.extend(y.tolist())
        y_pred += pred

print('accuracy_score :', accuracy_score(y_true, y_pred))
print('f1_score :', f1_score(y_true, y_pred))

accuracy_score : 0.5692934782608695
f1_score : 0.672858617131063


In [ ]:
samples = na_df['RawText'].sample(10).tolist()
samples

['두딸이 죽을 너무 좋아해서 자주 죽을 만들어 먹습니다. 재료들을 일일히 다져서 죽을 요리하는데 시간도 걸리고 귀찮을 때가 있습니다 죽제조기를 만나면서 얼마나 편해졌는지 몰라요 25분만 소요하면 뚝 딱 온갖 종류의 죽을 만들 수 있어서 너무 좋습니다 두딸이 좋아하는 전복죽 야채죽 닭죽 등 버튼하나만 눌러주면 마법같이 죽이 완성됩니다. 마트에서 사온 전복을 손질해서 바로 죽제조기에 넣었더니 맛나고 따끈따끈한 전복죽이 만들어졌답니다 나이많으신 어르신이나 죽을 좋아하는 아이들이 있는 집이라면 꼭 필요한 제품이라고 생각합니다',
 '코팅팬 수명이 다해서 다른 브랜드로 찾아보고 있던 참에 공간 활용이 큰 이 세트로 구매해봤어요 티타튬 소재 안심이구요 또 팬과 연결된 스텐레이스 부분이 노랗게 기름때 끼는거 정말 보기 싫었는데탈부착이라 깨끗하게 유지 될 거 같았어요 단점은 손잡이 맨 앞쪽 냄비 끼우는 곳 물때 안 끼게 닦는 거랑손잡이를 빼놓고 어디다 뒀는지 기억을 못 할 때가 있어요 그리고 손잡이 끼기 전에 가스불에 올려 놓고 난 후 생각나서 뜨거운 국물 담긴 냄비에 끼울 때 엄청 곤란했었네요 이번만 사용하고 다음에는 다른 브랜드로 갈아 타려고요 저 같이 깜박깜박하는 사람은 화상 입을까 염려돼요',
 '돌리고 돌리고 노래가 생각 나네요. 오늘도 전자레인지는 열심히 돌아갑니다.요즘 단호박 치즈 슬럿에 빠졌거든요.전자렌지만 있으면 간단하게 멋스러운 요리가 탄생합니다.출력조절기능도 가능해서 레토르트 식품 데울 때 쓰기 좋아요. 가끔 설명서에 쓰인대로 돌려도 덜 데워지거나 너무 데워지는 경우가 발생하는데 출력을 조절할수 있어서 그럴일이 없어요.온도 단계가 다양해서 전자렌지는 할수 없었던 그것 저온으로 숙성시키기가 간단하게 되서 굿이에요 . 요거트를 만들 수 있는 전자레인지라서 깜짝 놀랐어요.전자레인지는 그냥 급하게 음식이나 데우는용이라 생각했거든요.요리가 되고 요거트가 만들어지는 건 마법 같아요.',
 '잘못 산거 같네요.아무리 확인하고 보고 또 봐도 모기나 벌레가 죽어 있는 

In [ ]:
id2label = {
    0 : '부정',
    1 : '긍정'
}
@torch.no_grad()
def predict_review(texts, batch_size = 128):

    if isinstance(texts, str):
        texts = [texts]

    texts_norm = [normalize(t) for t in texts]

    # 2개의 모델을 평가모드 전환 clf, sbert
    sbert.eval()
    clf.eval()

    # 결과값
    result = []

    for idx in range(0, len(texts_norm), batch_size):
        batch_texts = texts_norm[idx : idx + batch_size]
  
        # sbert의 encode함수를 이용해서 임베딩 벡터 생성
        embs = sbert.encode(batch_texts, convert_to_tensor=True, normalize_embeddings=True)
        # embs를 clf모델을 이용하여 예측 확률 데이터를 생성
        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim = -1).tolist()

        for idx2, pred in enumerate(preds):
            prob = float(probs[idx2, pred])
     
            review = texts[idx + idx2]
            label = id2label[pred]

            result.append(
                {
                    'text' : review[:30],
                    'prob' : round(prob, 4),
                    'label' : label
                }
            )
    return result

In [ ]:
out_data = predict_review(samples)

In [ ]:
out_data

[{'text': '두딸이 죽을 너무 좋아해서 자주 죽을 만들어 먹습니다.', 'prob': 0.8229, 'label': '긍정'},
 {'text': '코팅팬 수명이 다해서 다른 브랜드로 찾아보고 있던 참에', 'prob': 0.7483, 'label': '부정'},
 {'text': '돌리고 돌리고 노래가 생각 나네요. 오늘도 전자레인지는', 'prob': 0.6071, 'label': '긍정'},
 {'text': '잘못 산거 같네요.아무리 확인하고 보고 또 봐도 모기나', 'prob': 0.8579, 'label': '부정'},
 {'text': 'OOO 쓰다가 노즐이 불편하여 구매했는데 다소 물이 적', 'prob': 0.7941, 'label': '부정'},
 {'text': '퀄리티가 엄청 좋은 건 아니에요.... 생각했던 것보다', 'prob': 0.8507, 'label': '부정'},
 {'text': '어머머 이건 사야하는 거예요 에스프레소샷잔도 필요했으니', 'prob': 0.8133, 'label': '긍정'},
 {'text': '벌써 겨울날씨네요 이웃님들 감기조심하세요 오늘 포스팅은', 'prob': 0.6941, 'label': '긍정'},
 {'text': '친정엄마가 김치냉장고는 꼭 이 브랜드로 너무 강조해서 ', 'prob': 0.6062, 'label': '긍정'},
 {'text': '애기 이유식 먹이려고 OOO 다지기 샀는데사자 마다 후', 'prob': 0.7855, 'label': '부정'}]